# Advice Agent
`adviceAgent.py` is the MAF implementation of the healthcare advice specialist.
It uses `tools/adviceApiTool.py` and `tools/webParseTool.py`.

Its general process is to:
1. Follow the system prompt.
2. Receive a prompt relating to healthcare, possibly with personal health parameters (age, sex, pregnancy, tobacco use, etc.).
3. Decide which tool to use based on the prompt rules.
4. Use the tool output to answer the user.

If specified, this agent can instead read general healthcare tips from Ted's Med Talk blog.
The logic of when to use each tool is primarily encoded in the system prompt.

## AG2 -> MAF Migration
| AG2 Pattern | MAF Equivalent |
|---|---|
| `AdviceAgent(BaseAgent)` | `build_advice_agent(prompts)` factory |
| `autogen` imports | `agent_framework` imports |
| `LLMConfig.from_json(...)` | `OpenAIChatClient(...)` |
| `UserProxyAgent` + `registerExecution(...)` | removed |
| `user_proxy.initiate_chat(...)` | `await advice_agent.run(...)` |
| `OAI_CONFIG_LIST.json` runtime loading | explicit client construction |

## Imports
Since this notebook lives in a `docs/` subfolder, we first adjust `sys.path` so imports resolve from the project root.


In [ ]:
import sys
from pathlib import Path

def setup_path():
    ai_root = Path().resolve().parent
    if str(ai_root) not in sys.path:
        sys.path.insert(0, str(ai_root))

setup_path()

Now we can define the MAF version of the advice agent.

This notebook stays consistent with the project's MAF 1.x migration and the OpenAI-compatible proxy configuration shown in `OAI_CONFIG_LIST.json`.


In [ ]:
from agent_framework import Agent
from agent_framework.openai import OpenAIChatClient

from tools.adviceApiTool import AdviceAPITool
from tools.webParseTool import HTMLParserTool
from utils import load_prompts

prompts = load_prompts()

## Shared Client
In AG2, the notebook loaded `LLMConfig` from `OAI_CONFIG_LIST.json` and passed it during `initiate_chat()`.
In MAF, we create the client directly and pass it into `Agent(...)`.

This version assumes the same backend shown elsewhere in the project:
- model: `qwen2.5:14b`
- base URL: `https://proxy/v1`
- API key: `ollama`

That keeps the migration consistent with the rest of the refactored code.


In [ ]:
client = OpenAIChatClient(
    model="qwen2.5:14b",
    base_url="https://proxy/v1",
    api_key="ollama",
    default_options={"temperature": 0.3, "tool_choice": "required"},
)

## Advice Agent Factory
The AG2 notebook used a `class AdviceAgent(BaseAgent)` pattern.
In the MAF refactor, the agent is created by a factory function instead.

Why this changed:
- `BaseAgent` in AG2 existed mostly to build `LLMConfig`, serialize tools, and register execution permissions.
- MAF does those parts directly through `Agent(...)` and native tool registration.
- So the class boilerplate stops being useful and the code gets smaller instead of more ceremonial. Rare win.


In [ ]:
def build_advice_agent(prompts: dict) -> Agent:
    return Agent(
        client=client,
        name="AdviceAgent",
        instructions=prompts["instructions"],
        tools=[AdviceAPITool(), HTMLParserTool()],
    )

## Running the Agent
In AG2, the test harness required a `UserProxyAgent`, `registerExecution(...)`, and `initiate_chat(...)`.
That extra loop is gone in MAF.

Now the agent runs directly:
- build the agent
- call `await advice_agent.run(message)`
- inspect the result

This also avoids the old auto-reply churn shown in the original notebook output, where the proxy kept sending blank turns until the run hit its limit.


In [ ]:
import asyncio

async def demo():
    advice_agent = build_advice_agent(prompts["advice"])

    result = await advice_agent.run(
        "I would like to get some healthcare advice. "
        "I am a 35 year old female, who is not pregnant, "
        "I am sexually active, and I do not smoke tobacco."
    )

    return result

response = asyncio.run(demo())
response

## Alternate Example
To test the webpage path instead of the health.gov API path, swap the message for a URL-based prompt like this:

```python
result = await advice_agent.run(
    "Can you check the health blog at http://tedmed/index.html and tell me what health advice it recommends?"
)
```

The system prompt should then steer the agent toward `HTMLParserTool()` instead of `AdviceAPITool()`.
